## 1. Import Libraries

Libraries for data processing and modeling.

- **pandas / numpy** → data handling and numerical operations  
- **train_test_split** → splitting data into training and testing sets  
- **LogisticRegression** → baseline model for fraud prediction  
- **classification_report / confusion_matrix** → evaluating model performance  

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

## 2. Load Data

Load the transaction dataset into a DataFrame.

The dataset contains transaction type, amount, account balances, and fraud labels.

In [2]:
df_raw = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")

## 3. Data Preparation

Select only the relevant columns for modeling.

This helps simplify the dataset and focus on key features related to fraud detection.

In [3]:
df_selected = df_raw[['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'isFraud']]

## 4. Feature Encoding

Machine learning models cannot understand text values (e.g., TRANSFER, CASH_OUT).

Using `get_dummies()`, we convert these categories into numeric columns (1 or 0 for each type).

The `drop_first=True` parameter removes one category to avoid duplicate information and improve model performance.

In [4]:
df_encoded = pd.get_dummies(df_selected, columns=['type'], drop_first=True)

## 5. Train/Test Split

The dataset is split into:
- training data → used by the model to learn patterns
- test data → used to evaluate performance on unseen data

This ensures the model is not just memorizing the data, but actually learning general patterns.

In [5]:
#axis=1 → applies the operation across columns (e.g., removing the 'isFraud' column)
X = df_encoded.drop('isFraud', axis=1)
y = df_encoded['isFraud']

## 6. Model Training

The dataset is split into training and test sets using an 80/20 ratio.

- The model learns patterns from **X_train and y_train**
- It is then evaluated on **X_test**, which it has never seen before

This step is important to ensure the model is not just memorizing the data, but actually learning patterns that can generalize to new, unseen transactions.

random_state ensures consistent results across runs.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 7. Model Evaluation

A Logistic Regression model is trained using X_train and y_train.

During training, the model learns the relationship between input features and the target variable by minimizing prediction errors.

The output shown (`LogisticRegression(...)`) confirms that the model has been successfully trained.  
It does not indicate model performance.

To evaluate performance, we use confusion matrix and classification report.

Since fraud cases are rare, recall and precision are more important than accuracy.

In [7]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## 8. Prediction

The trained model is used to make predictions on unseen test data.

For each transaction, the model outputs:
- 1 → fraud
- 0 → not fraud

These predictions (y_pred) are then compared with the actual values (y_test) to evaluate model performance.

In [8]:
y_pred = model.predict(X_test)

## Model Evaluation Guide

- **Precision** → how many predicted fraud cases are actually fraud  
  *(High → fewer false positives, Low → many normal transactions flagged as fraud)*

- **Recall** → how many actual fraud cases are detected  
  *(High → most fraud is captured, Low → many fraud cases are missed → high risk)*

- **F1-score** → balance between precision and recall  
  *(High → good overall balance, Low → model is biased toward either precision or recall)*

- **Accuracy** → overall correct predictions  
  *(Not very meaningful in imbalanced datasets like fraud detection)*

- **Macro avg** → average of metrics across classes (treats each class equally)  
  *(Useful to understand performance on minority class like fraud)*

- **Weighted avg** → average weighted by number of samples in each class  
  *(Dominated by majority class, may hide poor fraud detection performance)*

In [9]:
print(confusion_matrix(y_test, y_pred)) 
## [[TN  FP] -> [TP → correctly detected fraud ✔  TN → correctly identified normal transactions ✔
## [FN  TP]] -> [FP → false alarm (normal transaction flagged as fraud) ❗ FN → missed fraud 🚨 (most critical)]
print(classification_report(y_test, y_pred))

[[1270270     634]
 [     33    1587]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270904
           1       0.71      0.98      0.83      1620

    accuracy                           1.00   1272524
   macro avg       0.86      0.99      0.91   1272524
weighted avg       1.00      1.00      1.00   1272524



## Classification Report Interpretation

Each row represents a class:
- **0** → normal
- **1** → fraud

#### Class 0 - Normal
Near-perfect performance (expected due to majority class).

#### Class 1 - Fraud (Critical)
- **Precision = 0.71** → some false alarms
- **Recall = 0.98** → almost all fraud detected
- **F1-score = 0.83** → good balance
- **Support = 1,620** → number of fraud cases

#### Accuracy
Very high but not reliable due to class imbalance.

#### Macro Average
Treats classes equally → more realistic view.

#### Weighted Average
Dominated by normal class → may be overly optimistic.

## 10. Anomaly Detection Insight

Approximately 1% of transactions were flagged as anomalies by the Isolation Forest model.

These transactions may represent unusual patterns that are not explicitly labeled as fraud, indicating potential hidden risks.

Combining anomaly detection with fraud prediction improves overall detection coverage and supports proactive risk monitoring.

In [11]:
from sklearn.ensemble import IsolationForest

# 1. Sample al
sample_df = df_encoded.sample(200000, random_state=42)

X_sample = sample_df.drop('isFraud', axis=1).values

# 2. Model fit
iso = IsolationForest(contamination=0.01, random_state=42)
iso.fit(X_sample)

# 3. Full data predict
X_full = df_encoded.drop('isFraud', axis=1).values

df_model = df_encoded.copy()

df_model['anomaly_score'] = iso.predict(X_full)
df_model['is_anomaly'] = (df_model['anomaly_score'] == -1).astype(int)

print(df_model['is_anomaly'].value_counts())

0    6297044
1      65576
Name: is_anomaly, dtype: int64


## Business Interpretation

- The model successfully detects the majority of fraud cases (~98% recall), ensuring that very few fraudulent transactions are missed.
- Some legitimate transactions are flagged as fraud, indicating a trade-off between precision and recall.
- In addition to model predictions, anomaly detection identifies approximately 1% of transactions as unusual, highlighting potential hidden risks beyond labeled fraud cases.

## What this means

- Fraud detection is strong and reliable, minimizing the risk of undetected fraudulent activity.
- The system not only captures known fraud patterns but also detects abnormal transaction behavior that may indicate emerging or unknown fraud types.
- Transactions flagged either by the model or anomaly detection should be prioritized for further investigation.
- This hybrid approach supports proactive risk management rather than reactive fraud detection.

## Conclusion

The combined use of machine learning (Logistic Regression) and anomaly detection (Isolation Forest) provides a more robust fraud detection framework.

While the model performs well as a baseline solution, integrating anomaly detection enhances coverage and enables identification of suspicious behavior beyond predefined fraud patterns.

This approach can serve as a scalable foundation for building advanced, AI-powered fraud monitoring systems.

In [12]:
df_model.to_csv("../data/fraud_analysis_enriched.csv", index=False)